# QDM Phase 2B — BNB-only add-on notebook

This notebook adds **bitsandbytes INT8** and **bitsandbytes NF4** conditions only.

It does **not** rerun RTN or pruning. It:
1. Loads the existing Phase 2B output folder.
2. Loads HF FP16 reference model + BNB quantized models.
3. Computes streaming SAE feature correlations without storing giant feature matrices.
4. Saves BNB summaries/per-feature CSVs into the same Phase 2B folder.
5. Merges BNB rows into `phase2b_summary_final_with_bnb.csv`.

Run this after installing bitsandbytes.

In [ ]:
# Optional install cell.
# If bitsandbytes is already installed, you can skip this.
# If install fails due to Debian/system packages, run it in terminal or restart kernel after install.

import sys
print("Python:", sys.executable)

# Uncomment if needed:
# !{sys.executable} -m pip install -U --ignore-installed "bitsandbytes>=0.46.1" accelerate rich

In [ ]:
# Imports and setup

import os
import gc
import json
import math
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sae_lens import SAE

try:
    import bitsandbytes as bnb
    print("bitsandbytes import OK")
except Exception as e:
    print("bitsandbytes import failed:", repr(e))
    raise

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
# Configuration

RUN_MODE = "full"   # "test" or "full"

MODEL_NAME = "EleutherAI/pythia-70m-deduped"
SAE_RELEASE = "pythia-70m-deduped-res-sm"
LAYER = 4
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"

SEQ_LEN = 512
TOKEN_BUDGET = 20_000 if RUN_MODE == "test" else 200_000
BATCH_SIZE = 4

FIRING_THRESHOLD = 0.001

OUTPUT_DIR = Path(f"phase2b_outputs_{RUN_MODE}_{TOKEN_BUDGET//1000}k_L{LAYER}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("TOKEN_BUDGET:", TOKEN_BUDGET)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

In [ ]:
# Hugging Face login is optional for public Pythia, but helpful for rate limits.
# Uncomment if needed.

# from huggingface_hub import login
# import getpass, os
# hf_token = getpass.getpass("Paste your Hugging Face token: ")
# login(token=hf_token)
# os.environ["HF_TOKEN"] = hf_token
# os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
# print("HF login complete.")

In [ ]:
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def build_tokens_2d(tokenizer, token_budget, seq_len):
    # Test mode uses smaller split; full mode uses train to guarantee enough tokens.
    split = "test" if token_budget <= 20_000 else "train"
    print(f"Using WikiText-2 split: {split}")

    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split=split)

    # Keep all non-empty lines. Do NOT use len > 100 filter.
    full_text = "\n\n".join(x for x in ds["text"] if x.strip())
    print(f"Total characters: {len(full_text):,}")

    token_ids = tokenizer.encode(full_text, add_special_tokens=False)
    tokens = torch.tensor(token_ids, dtype=torch.long)

    print(f"Total available tokens: {tokens.shape[0]:,}")

    usable_tokens = min(token_budget, tokens.shape[0])
    n_seqs = usable_tokens // seq_len
    usable_tokens = n_seqs * seq_len

    if usable_tokens == 0:
        raise RuntimeError("Not enough tokens for one sequence.")

    tokens_2d = tokens[:usable_tokens].reshape(n_seqs, seq_len).to(DEVICE)

    print(f"Using tokens: {usable_tokens:,}")
    print(f"tokens_2d shape: {tuple(tokens_2d.shape)}")

    return tokens_2d

In [ ]:
# Load tokenizer and tokens

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokens_2d = build_tokens_2d(tokenizer, TOKEN_BUDGET, SEQ_LEN)

In [ ]:
# Load SAE

sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=HOOK_NAME,
    device=DEVICE,
)

sae.eval()

print("SAE loaded:", SAE_RELEASE, HOOK_NAME)
print("d_in:", sae.cfg.d_in)
print("d_sae:", sae.cfg.d_sae)

In [ ]:
# Model loaders

def load_hf_fp16_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map={"": 0} if DEVICE == "cuda" else None,
    )
    model.eval()
    model.config.use_cache = False
    return model


def load_hf_bnb_int8_model():
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        device_map={"": 0} if DEVICE == "cuda" else None,
    )
    model.eval()
    model.config.use_cache = False
    return model


def load_hf_bnb_nf4_model():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        device_map={"": 0} if DEVICE == "cuda" else None,
    )
    model.eval()
    model.config.use_cache = False
    return model

In [ ]:
def compute_hf_perplexity(model, tokens_2d, batch_size=4, desc="HF perplexity"):
    losses = []

    for i in tqdm(range(0, tokens_2d.shape[0], batch_size), desc=desc):
        batch = tokens_2d[i:i + batch_size]

        with torch.no_grad():
            out = model(input_ids=batch, labels=batch)
            loss = out.loss

        losses.append(float(loss.detach().cpu()))

    avg_loss = float(np.mean(losses))
    ppl = float(np.exp(avg_loss))
    return ppl, avg_loss


def get_hf_layer_acts(model, batch, layer_idx):
    # HF hidden_states[0] = embedding output
    # HF hidden_states[layer_idx + 1] = post-block layer_idx output
    with torch.no_grad():
        out = model(
            input_ids=batch,
            output_hidden_states=True,
            use_cache=False,
        )

    acts = out.hidden_states[layer_idx + 1].detach()
    acts = acts.reshape(-1, acts.shape[-1])
    return acts


def encode_sae_cpu(sae, acts):
    with torch.no_grad():
        feats = sae.encode(acts.to(DEVICE).float())
    return feats.detach().cpu().to(torch.float64)

In [ ]:
def streaming_hf_condition(
    ref_model,
    test_model,
    sae,
    tokens_2d,
    layer_idx,
    condition_name,
    bits,
    ppl,
    loss,
    ppl_ref,
    batch_size=4,
    firing_threshold=0.001,
):
    '''
    Compare HF FP16 reference model vs a test HF model, batch-by-batch.
    Saves no token x feature matrices.
    '''
    d_sae = sae.cfg.d_sae

    sum_x = torch.zeros(d_sae, dtype=torch.float64)
    sum_y = torch.zeros(d_sae, dtype=torch.float64)
    sum_x2 = torch.zeros(d_sae, dtype=torch.float64)
    sum_y2 = torch.zeros(d_sae, dtype=torch.float64)
    sum_xy = torch.zeros(d_sae, dtype=torch.float64)

    fire_count = torch.zeros(d_sae, dtype=torch.float64)
    sum_x_activation = torch.zeros(d_sae, dtype=torch.float64)
    max_x_activation = torch.zeros(d_sae, dtype=torch.float64)

    total_positions = 0

    for i in tqdm(range(0, tokens_2d.shape[0], batch_size), desc=f"Streaming {condition_name}"):
        batch = tokens_2d[i:i + batch_size]

        acts_ref = get_hf_layer_acts(ref_model, batch, layer_idx)
        feats_ref = encode_sae_cpu(sae, acts_ref)

        acts_test = get_hf_layer_acts(test_model, batch, layer_idx)
        feats_test = encode_sae_cpu(sae, acts_test)

        x = feats_ref
        y = feats_test

        sum_x += x.sum(dim=0)
        sum_y += y.sum(dim=0)
        sum_x2 += (x ** 2).sum(dim=0)
        sum_y2 += (y ** 2).sum(dim=0)
        sum_xy += (x * y).sum(dim=0)

        fire_count += (x > 0).sum(dim=0)
        sum_x_activation += x.sum(dim=0)
        max_x_activation = torch.maximum(max_x_activation, x.max(dim=0).values)

        total_positions += x.shape[0]

        del acts_ref, acts_test, feats_ref, feats_test, x, y
        free_memory()

    n = total_positions

    numerator = sum_xy - (sum_x * sum_y / n)
    denom_x = sum_x2 - (sum_x ** 2 / n)
    denom_y = sum_y2 - (sum_y ** 2 / n)
    denominator = torch.sqrt(torch.clamp(denom_x * denom_y, min=1e-12))

    corr = torch.clamp(numerator / denominator, -1.0, 1.0)

    firing_rate = fire_count / n
    mean_activation = sum_x_activation / n
    active_mask = firing_rate > firing_threshold
    active_corrs = corr[active_mask]

    if active_corrs.numel() == 0:
        raise RuntimeError("No active features found; lower firing_threshold.")

    survived = (active_corrs > 0.9).double().mean().item() * 100
    degraded = ((active_corrs > 0.5) & (active_corrs <= 0.9)).double().mean().item() * 100
    damaged = (active_corrs < 0.5).double().mean().item() * 100

    summary = {
        "condition": condition_name,
        "bits": bits,
        "n_tokens": int(n),
        "n_total_features": int(d_sae),
        "n_active_features": int(active_mask.sum().item()),
        "mean_corr": float(active_corrs.mean().item()),
        "median_corr": float(active_corrs.median().item()),
        "survived_>0.9_pct": float(survived),
        "degraded_0.5_0.9_pct": float(degraded),
        "damaged_<0.5_pct": float(damaged),
        "layer": int(layer_idx),
        "perplexity": float(ppl),
        "loss": float(loss),
        "ppl_delta_pct": float((ppl / ppl_ref - 1) * 100),
        "reference": "HF_FP16",
        "method_family": "bitsandbytes",
    }

    per_feature = pd.DataFrame({
        "feature_id": np.arange(d_sae),
        "condition": condition_name,
        "bits": bits,
        "corr": corr.numpy(),
        "firing_rate": firing_rate.numpy(),
        "mean_activation": mean_activation.numpy(),
        "max_activation": max_x_activation.numpy(),
        "active": active_mask.numpy(),
        "survived_>0.9": ((corr > 0.9) & active_mask).numpy(),
        "damaged_<0.5": ((corr < 0.5) & active_mask).numpy(),
        "layer": int(layer_idx),
        "reference": "HF_FP16",
        "method_family": "bitsandbytes",
    })

    return summary, per_feature

In [ ]:
# Load HF FP16 reference

free_memory()

print("Loading HF FP16 reference model...")
hf_ref = load_hf_fp16_model()

ppl_hf_fp16, loss_hf_fp16 = compute_hf_perplexity(
    hf_ref,
    tokens_2d,
    batch_size=BATCH_SIZE,
    desc="HF FP16 perplexity",
)

print(f"HF FP16 ppl={ppl_hf_fp16:.3f}, loss={loss_hf_fp16:.4f}")

In [ ]:
# Run BNB INT8 only

bnb_summaries = []
bnb_per_feature_files = {}

try:
    print("Loading BNB INT8 model...")
    bnb_int8 = load_hf_bnb_int8_model()

    ppl_int8, loss_int8 = compute_hf_perplexity(
        bnb_int8,
        tokens_2d,
        batch_size=BATCH_SIZE,
        desc="BNB INT8 perplexity",
    )

    print(f"BNB INT8 ppl={ppl_int8:.3f}, delta={(ppl_int8/ppl_hf_fp16 - 1)*100:+.2f}%")

    summary_int8, pf_int8 = streaming_hf_condition(
        ref_model=hf_ref,
        test_model=bnb_int8,
        sae=sae,
        tokens_2d=tokens_2d,
        layer_idx=LAYER,
        condition_name="BNB_INT8",
        bits=8,
        ppl=ppl_int8,
        loss=loss_int8,
        ppl_ref=ppl_hf_fp16,
        batch_size=BATCH_SIZE,
        firing_threshold=FIRING_THRESHOLD,
    )

    summary_path = OUTPUT_DIR / "BNB_INT8_summary.csv"
    per_feature_path = OUTPUT_DIR / "BNB_INT8_per_feature.csv"

    pd.DataFrame([summary_int8]).to_csv(summary_path, index=False)
    pf_int8.to_csv(per_feature_path, index=False)

    bnb_summaries.append(summary_int8)
    bnb_per_feature_files["BNB_INT8"] = str(per_feature_path)

    print("Saved:", summary_path)
    print("Saved:", per_feature_path)

    del bnb_int8, pf_int8
    free_memory()

except Exception as e:
    err_path = OUTPUT_DIR / "bnb_int8_error_retry.txt"
    with open(err_path, "w") as f:
        f.write(repr(e))
    print("BNB INT8 failed:", repr(e))
    print("Saved error:", err_path)

In [ ]:
# Run BNB NF4 only

try:
    print("Loading BNB NF4 model...")
    bnb_nf4 = load_hf_bnb_nf4_model()

    ppl_nf4, loss_nf4 = compute_hf_perplexity(
        bnb_nf4,
        tokens_2d,
        batch_size=BATCH_SIZE,
        desc="BNB NF4 perplexity",
    )

    print(f"BNB NF4 ppl={ppl_nf4:.3f}, delta={(ppl_nf4/ppl_hf_fp16 - 1)*100:+.2f}%")

    summary_nf4, pf_nf4 = streaming_hf_condition(
        ref_model=hf_ref,
        test_model=bnb_nf4,
        sae=sae,
        tokens_2d=tokens_2d,
        layer_idx=LAYER,
        condition_name="BNB_NF4",
        bits=4,
        ppl=ppl_nf4,
        loss=loss_nf4,
        ppl_ref=ppl_hf_fp16,
        batch_size=BATCH_SIZE,
        firing_threshold=FIRING_THRESHOLD,
    )

    summary_path = OUTPUT_DIR / "BNB_NF4_summary.csv"
    per_feature_path = OUTPUT_DIR / "BNB_NF4_per_feature.csv"

    pd.DataFrame([summary_nf4]).to_csv(summary_path, index=False)
    pf_nf4.to_csv(per_feature_path, index=False)

    bnb_summaries.append(summary_nf4)
    bnb_per_feature_files["BNB_NF4"] = str(per_feature_path)

    print("Saved:", summary_path)
    print("Saved:", per_feature_path)

    del bnb_nf4, pf_nf4
    free_memory()

except Exception as e:
    err_path = OUTPUT_DIR / "bnb_nf4_error_retry.txt"
    with open(err_path, "w") as f:
        f.write(repr(e))
    print("BNB NF4 failed:", repr(e))
    print("Saved error:", err_path)

In [ ]:
# Save BNB-only summary

if len(bnb_summaries) > 0:
    bnb_df = pd.DataFrame(bnb_summaries)
    bnb_summary_path = OUTPUT_DIR / "phase2b_bnb_only_summary.csv"
    bnb_df.to_csv(bnb_summary_path, index=False)
    display(bnb_df)
    print("Saved:", bnb_summary_path)

    with open(OUTPUT_DIR / "phase2b_bnb_per_feature_files.json", "w") as f:
        json.dump(bnb_per_feature_files, f, indent=2)
else:
    print("No BNB conditions succeeded.")

In [ ]:
# Merge BNB rows into existing Phase 2B final summary

existing_summary_path = OUTPUT_DIR / "phase2b_summary_final.csv"
merged_summary_path = OUTPUT_DIR / "phase2b_summary_final_with_bnb.csv"

if existing_summary_path.exists() and len(bnb_summaries) > 0:
    existing = pd.read_csv(existing_summary_path)
    bnb_df = pd.DataFrame(bnb_summaries)

    # Avoid duplicate BNB rows if rerunning this cell
    existing = existing[~existing["condition"].isin(bnb_df["condition"].tolist())]

    merged = pd.concat([existing, bnb_df], ignore_index=True, sort=False)

    # Nice ordering where possible
    order = [
        "FP16 baseline",
        "RTN_INT8", "RTN_INT7", "RTN_INT6", "RTN_INT5", "RTN_INT4",
        "Magnitude_pruning_matched_INT6",
        "BNB_INT8", "BNB_NF4",
    ]
    merged["__order"] = merged["condition"].apply(lambda x: order.index(x) if x in order else 999)
    merged = merged.sort_values(["__order"]).drop(columns=["__order"])

    merged.to_csv(merged_summary_path, index=False)

    pd.set_option("display.max_columns", None)
    pd.set_option("display.float_format", "{:.3f}".format)
    display(merged)
    print("Saved merged summary:", merged_summary_path)

elif not existing_summary_path.exists():
    print("Existing summary not found:", existing_summary_path)
    print("BNB summary still saved separately if BNB succeeded.")
else:
    print("No BNB rows to merge.")

In [ ]:
# List output files

for f in sorted(OUTPUT_DIR.glob("*")):
    print(f)